# Weakly Supervised TAR Workflow (Full)

Includes:
- Weak supervision labeling (0 = Irrelevant, 1 = Relevant)
- Train / Validation / Holdout split
- 3 models: Logistic Regression, Linear SVM, DistilBERT
- Prints **Validation** metrics (Accuracy, Precision, Recall, F1) after each model trains
- Prints **Gold test** metrics (Accuracy, Precision, Recall, F1)
- Draws **Gold test** confusion matrices
- Saves tables + images to a timestamped folder in Google Drive


## 0) Setup: Mount Drive + install packages + imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install/upgrade transformers stack (fixes older Colab versions)
!pip -q install -U transformers datasets accelerate

import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import transformers, datasets, accelerate
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)


## 1) Output folder in Google Drive

In [ ]:
RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"/content/drive/MyDrive/enron/enron_tar_outputs/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving outputs to:", OUT_DIR)


## 2) Load dataset

In [ ]:
CSV_PATH = "/content/drive/MyDrive/enron/enron.csv"  # <-- change if needed
df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape)
df.head(2)


## 3) Preprocessing functions

In [ ]:
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_text(df, subject_col="Subject", message_col="Message"):
    subj = df[subject_col].fillna("").astype(str)
    msg = df[message_col].fillna("").astype(str)
    return (subj + " " + msg).map(normalize_text)

def deduplicate(df, text_col="text", id_col="Message-ID"):
    df = df.copy()
    if id_col in df.columns:
        return df.drop_duplicates(subset=[id_col])
    df["_text_hash"] = pd.util.hash_pandas_object(df[text_col], index=False)
    df = df.drop_duplicates(subset=["_text_hash"]).drop(columns=["_text_hash"])
    return df


## 4) Weak supervision keywords + labeling functions

In [ ]:
RELEVANT_KEYWORDS = [
    "accounting", "agenda", "agreement", "approval", "audit",
    "board", "bonus",
    "capacity", "compliance", "concern", "contract", "counterparty", "credit",
    "debt", "decision", "derivatives", "directive", "disclosure", "discussion",
    "earnings", "electricity", "employee", "exposure", "executive",
    "financials", "finance", "forecast",
    "gas",
    "hedge", "hedging", "hire", "hr",
    "investigation", "issue",
    "leadership", "legal", "liability", "loss",
    "management", "margin", "mark-to-market", "market", "meeting",
    "numbers",
    "off-balance-sheet",
    "performance", "personnel", "pipeline", "plan", "policy", "portfolio",
    "position", "power", "pricing", "problem", "profit", "proposal", "promotion",
    "reorganization", "report", "revenue", "review", "review by counsel", "risk",
    "spe", "staff", "strategy", "supply",
    "team", "trading", "transaction",
    "update",
    "valuation"
]

def keyword_label(text: str) -> int:
    return 1 if any(k in text for k in RELEVANT_KEYWORDS) else 0

def apply_weak_labels(text_series: pd.Series) -> np.ndarray:
    return np.array([keyword_label(t) for t in text_series], dtype=int)


## 5) Prepare data (text + weak labels)

In [ ]:
df = df.copy()
df["text"] = build_text(df, subject_col="Subject", message_col="Message")
df = deduplicate(df, text_col="text", id_col="Message-ID")
df["weak_label"] = apply_weak_labels(df["text"])

print("Rows after dedup:", len(df))
print("Weak relevant rate:", df["weak_label"].mean())
df[["Subject","weak_label"]].head(5)


## 6) Train / Validation / Holdout split

In [ ]:
RANDOM_STATE = 42

# Holdout pool for gold sampling (20%)
trainval_df, holdout_pool = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["weak_label"] if df["weak_label"].nunique() > 1 else None
)

# Train/Val split from remaining (Val = 20% of trainval => 16% overall)
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=trainval_df["weak_label"] if trainval_df["weak_label"].nunique() > 1 else None
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Holdout:", holdout_pool.shape)


## 7) Export a gold sample for manual labeling

In [ ]:
N_GOLD = 1000  # number of emails to label manually

gold_sample = holdout_pool.sample(n=min(N_GOLD, len(holdout_pool)), random_state=RANDOM_STATE).copy()

keep_cols = [c for c in ["Message-ID","Date","From","To","Cc","Bcc","Subject","Message","text","weak_label"] if c in gold_sample.columns]
gold_export = gold_sample[keep_cols].copy()
gold_export["gold_label"] = ""  # fill with 0/1 manually

GOLD_TO_LABEL_PATH = "/content/gold_test_to_label.csv"
gold_export.to_csv(GOLD_TO_LABEL_PATH, index=False)

print("Saved gold file for manual labeling to:", GOLD_TO_LABEL_PATH)
gold_export.head(3)


## 8) TF-IDF features

In [ ]:
tfidf = TfidfVectorizer(min_df=3, max_df=0.9, ngram_range=(1,2))

X_train = tfidf.fit_transform(train_df["text"])
y_train = train_df["weak_label"].astype(int).values

X_val = tfidf.transform(val_df["text"])
y_val = val_df["weak_label"].astype(int).values

print("TF-IDF ready. X_train shape:", X_train.shape)


## 9) Metrics + confusion matrix helpers

In [ ]:
def metrics_dict(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

def save_confusion_png(y_true, y_pred, title, out_png):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

VAL_RESULTS = {}
GOLD_RESULTS = {}


## 10) Model 1: Logistic Regression (train + VALIDATION metrics)

In [ ]:
logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_train, y_train)

val_pred_lr = logreg.predict(X_val)
VAL_RESULTS["Logistic Regression (TF-IDF)"] = metrics_dict(y_val, val_pred_lr)

print("Validation metrics — Logistic Regression (TF-IDF)")
print(VAL_RESULTS["Logistic Regression (TF-IDF)"])


## 11) Model 2: Linear SVM (train + VALIDATION metrics)

In [ ]:
svm = LinearSVC()
svm.fit(X_train, y_train)

val_pred_svm = svm.predict(X_val)
VAL_RESULTS["Linear SVM (TF-IDF)"] = metrics_dict(y_val, val_pred_svm)

print("Validation metrics — Linear SVM (TF-IDF)")
print(VAL_RESULTS["Linear SVM (TF-IDF)"])


## 12) Model 3: DistilBERT (train + VALIDATION metrics)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_hf = Dataset.from_pandas(train_df[["text","weak_label"]].rename(columns={"weak_label":"labels"}))
val_hf = Dataset.from_pandas(val_df[["text","weak_label"]].rename(columns={"weak_label":"labels"}))

train_hf = train_hf.map(tokenize_batch, batched=True)
val_hf = val_hf.map(tokenize_batch, batched=True)

train_hf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
val_hf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="/content/bert_out",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
)

trainer.train()

# Validation metrics immediately after training
val_logits = trainer.predict(val_hf).predictions
val_pred_bert = np.argmax(val_logits, axis=1)
VAL_RESULTS["DistilBERT"] = metrics_dict(y_val, val_pred_bert)

print("Validation metrics — DistilBERT")
print(VAL_RESULTS["DistilBERT"])


## 12b) CNN (TextCNN) baseline

This section adds a simple CNN text classifier using Keras. It trains on the same weak labels, prints **validation** metrics immediately after training, evaluates on the **Gold test** set, saves a **Gold confusion matrix**, and includes CNN in the saved comparison tables.

In [ ]:
# Keras / TensorFlow for CNN
!pip -q install tensorflow

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout


In [ ]:
# Tokenization for CNN
MAX_WORDS = 20000
MAX_LEN = 200

tokenizer_cnn = Tokenizer(num_words=MAX_WORDS)
tokenizer_cnn.fit_on_texts(train_df["text"])

X_train_cnn = tokenizer_cnn.texts_to_sequences(train_df["text"])
X_val_cnn = tokenizer_cnn.texts_to_sequences(val_df["text"])

X_train_cnn = pad_sequences(X_train_cnn, maxlen=MAX_LEN)
X_val_cnn = pad_sequences(X_val_cnn, maxlen=MAX_LEN)

y_train_cnn = y_train
y_val_cnn = y_val

print("CNN inputs ready:", X_train_cnn.shape, X_val_cnn.shape)


In [ ]:
# Build CNN model
input_layer = Input(shape=(MAX_LEN,))
embedding = Embedding(MAX_WORDS, 128)(input_layer)

conv = Conv1D(128, 5, activation="relu")(embedding)
pool = GlobalMaxPooling1D()(conv)

dense = Dense(64, activation="relu")(pool)
drop = Dropout(0.5)(dense)

output = Dense(1, activation="sigmoid")(drop)

cnn_model = Model(inputs=input_layer, outputs=output)

cnn_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

cnn_model.summary()


In [ ]:
# Train CNN + VALIDATION metrics
cnn_model.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=3,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn),
    verbose=1
)

val_pred_cnn = (cnn_model.predict(X_val_cnn) > 0.5).astype(int).ravel()
VAL_RESULTS["CNN"] = metrics_dict(y_val_cnn, val_pred_cnn)

print("Validation metrics — CNN")
print(VAL_RESULTS["CNN"])


## 13) Validation comparison table (ALL models) + save to Drive

In [ ]:
val_table = pd.DataFrame([{"model": k, **v} for k, v in VAL_RESULTS.items()])     .sort_values(by=["recall","f1"], ascending=False).reset_index(drop=True)

print("VALIDATION COMPARISON TABLE (ALL MODELS)")
display(val_table)

val_csv = os.path.join(OUT_DIR, "validation_metrics_all_models.csv")
val_table.to_csv(val_csv, index=False)
print("Saved:", val_csv)


## 14) Load manually labeled GOLD test set

You must create this file by copying `gold_test_to_label.csv`, filling `gold_label` with 0/1, and saving as `gold_test_labeled.csv`.

In [ ]:
GOLD_LABELED_PATH = "/content/drive/MyDrive/enron/gold_test_to_label.csv"  # <-- change if needed
gold_df = pd.read_csv(GOLD_LABELED_PATH)

# Ensure text exists
if "text" not in gold_df.columns:
    gold_df["text"] = (gold_df["Subject"].fillna("") + " " + gold_df["Message"].fillna("")).map(normalize_text)
else:
    gold_df["text"] = gold_df["text"].fillna("").astype(str).map(normalize_text)

# Clean labels and keep only valid 0/1 rows
gold_df["gold_label"] = gold_df["gold_label"].astype(str).str.strip()
gold_df = gold_df[gold_df["gold_label"].isin(["0","1"])].copy()

y_gold = gold_df["gold_label"].astype(int).values
X_gold = tfidf.transform(gold_df["text"])

print("Gold test size:", len(gold_df), "Relevant rate:", float(y_gold.mean()))


## 15) GOLD test metrics (each model) + comparison table + save to Drive

In [ ]:
# CNN GOLD test evaluation
X_gold_cnn = tokenizer_cnn.texts_to_sequences(gold_df["text"])
X_gold_cnn = pad_sequences(X_gold_cnn, maxlen=MAX_LEN)
y_gold_cnn = y_gold

gold_pred_cnn = (cnn_model.predict(X_gold_cnn) > 0.5).astype(int).ravel()
GOLD_RESULTS["CNN"] = metrics_dict(y_gold_cnn, gold_pred_cnn)

print("Gold test metrics — CNN")
print(GOLD_RESULTS["CNN"])


In [ ]:
# Logistic Regression
gold_pred_lr = logreg.predict(X_gold)
GOLD_RESULTS["Logistic Regression (TF-IDF)"] = metrics_dict(y_gold, gold_pred_lr)

# Linear SVM
gold_pred_svm = svm.predict(X_gold)
GOLD_RESULTS["Linear SVM (TF-IDF)"] = metrics_dict(y_gold, gold_pred_svm)

# DistilBERT
gold_hf = Dataset.from_pandas(gold_df[["text"]].copy())
gold_hf = gold_hf.map(tokenize_batch, batched=True)
gold_hf.set_format(type="torch", columns=["input_ids","attention_mask"])

gold_logits = trainer.predict(gold_hf).predictions
gold_pred_bert = np.argmax(gold_logits, axis=1)
GOLD_RESULTS["DistilBERT"] = metrics_dict(y_gold, gold_pred_bert)

# Table
gold_table = pd.DataFrame([{"model": k, **v} for k, v in GOLD_RESULTS.items()])     .sort_values(by=["recall","f1"], ascending=False).reset_index(drop=True)

print("GOLD TEST COMPARISON TABLE (ALL MODELS)")
display(gold_table)

gold_csv = os.path.join(OUT_DIR, "gold_test_metrics_all_models.csv")
gold_table.to_csv(gold_csv, index=False)
print("Saved:", gold_csv)

# Combined metrics (validation + gold)
val_table2 = val_table.copy()
val_table2.insert(0, "split", "validation")

gold_table2 = gold_table.copy()
gold_table2.insert(0, "split", "gold_test")

all_table = pd.concat([val_table2, gold_table2], ignore_index=True)
all_csv = os.path.join(OUT_DIR, "all_metrics_validation_and_gold.csv")
all_table.to_csv(all_csv, index=False)
print("Saved:", all_csv)


## 16) Confusion matrices (GOLD test) + save PNGs to Drive

In [ ]:
cm_lr_path = os.path.join(OUT_DIR, "cm_gold_logreg.png")
cm_svm_path = os.path.join(OUT_DIR, "cm_gold_svm.png")
cm_bert_path = os.path.join(OUT_DIR, "cm_gold_distilbert.png")

save_confusion_png(y_gold, gold_pred_lr, "Gold Confusion Matrix — Logistic Regression", cm_lr_path)
save_confusion_png(y_gold, gold_pred_svm, "Gold Confusion Matrix — Linear SVM", cm_svm_path)
save_confusion_png(y_gold, gold_pred_bert, "Gold Confusion Matrix — DistilBERT", cm_bert_path)

print("Saved confusion matrices:")
print(cm_lr_path)
print(cm_svm_path)
print(cm_bert_path)


In [ ]:
# CNN confusion matrix (Gold test)
cm_cnn_path = os.path.join(OUT_DIR, "cm_gold_cnn.png")
save_confusion_png(y_gold_cnn, gold_pred_cnn, "Gold Confusion Matrix — CNN", cm_cnn_path)
print("Saved:", cm_cnn_path)


## 17) Done

In [ ]:
print("DONE.")
print("All outputs saved to:", OUT_DIR)
